# Modeling, Segmentation & Evaluation

## Overview
This notebook focuses on building, comparing, and evaluating machine learning models for the Bank Marketing dataset. Building on the preprocessing and feature engineering steps from previous notebooks, we now develop predictive models to estimate the likelihood of a client subscribing to a term deposit.

In addition to a traditional global modeling approach, this notebook explores a segmentation-based strategy using clusters derived earlier. By comparing a single global model against cluster-specific models, we aim to understand whether segment-level specialization leads to improved predictive performance and better business outcomes.



## Objectives
- Compare multiple candidate model architectures using the training and validation sets  
- Select a primary modeling approach based on validation performance  
- Train a global model using all available data and evaluate performance across clusters  
- Train segment-specific models for each cluster and compare against the global model  
- Perform hyperparameter tuning for both global and segment-specific models using validation data  
- Train final models using selected hyperparameters  
- Apply probability calibration to improve predicted probabilities  
- Conduct cost-sensitive analysis using calibrated probabilities and different decision thresholds  
- Evaluate final models on a held-out test set, including overall and cluster-level performance  



## Dataset Description
The modeling process uses the preprocessed dataset generated in the previous notebook, which includes engineered features such as categorical encodings, binary indicators, and interaction terms. Additionally, each observation is assigned to a cluster derived from unsupervised learning, enabling both global and segment-specific modeling approaches.

The data is split into three subsets:

- **Training set**: used to fit models  
- **Validation set**: used for model selection, hyperparameter tuning, and calibration  
- **Test set**: held out for final unbiased evaluation  

The target variable represents whether a client subscribed to a term deposit.



## Key Considerations
- **Model comparison**: Different model architectures are evaluated to identify the most suitable approach for the problem  
- **Segmentation strategy**: Cluster-based segmentation allows us to investigate heterogeneity in client behavior and model performance  
- **Fairness and consistency**: Performance is evaluated across clusters to detect potential disparities  
- **Overfitting prevention**: Validation data is strictly used for model selection and hyperparameter tuning  
- **Probability calibration**: Raw model probabilities may be poorly calibrated, which can negatively impact decision-making  
- **Business alignment**: Model evaluation incorporates cost-sensitive analysis, considering trade-offs between false positives (e.g., unnecessary calls) and false negatives (missed opportunities)  
- **Final evaluation integrity**: The test set remains untouched until the final evaluation step  



## Outcome
By the end of this notebook, we will have:
- Selected and trained a primary global model  
- Developed segment-specific models tailored to different client clusters  
- Tuned and calibrated all models for improved predictive reliability  
- Compared global and segmented approaches from both predictive and business perspectives  
- Identified optimal decision thresholds based on expected cost  
- Produced a final evaluation on the test set, including overall performance, cluster-level insights, and cost-based metrics  

In [2]:
import pandas as pd
import lightgbm as lgb
import optuna
import joblib

from pathlib import Path
from sklearn.model_selection import ParameterGrid
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import average_precision_score

from utils.utils import load_dataset, evaluate_model, evaluate_from_probs

RANDOM_STATE = 705

In [3]:
# quarto preview 03_training.ipynb --to pdf
# quarto render 03_training.ipynb
# black 03_training.ipynb

## 1. Global architecture selection

In this section, we evaluate a set of candidate model architectures to determine which will be used in the subsequent comparison between a global model (trained on all observations) and segment-specific models (trained per cluster).

We begin by training and assessing five baseline architectures using their default configurations:

1. Logistic Regression  
2. Random Forest  
3. XGBoost  
4. LightGBM  
5. CatBoost  

The goal is to identify a strong primary model that balances predictive performance and generalization before proceeding to more advanced analysis.

In [4]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# from sklearn.metrics import (
#     average_precision_score,
#     f1_score,
#     precision_score,
#     recall_score,
# )

from xgboost import XGBClassifier
import lightgbm as lgb
from catboost import CatBoostClassifier

Because the outcome variable is highly imbalanced, we apply class weighting across all models to penalize misclassification of subscribers more heavily than non‑subscribers. For tree‑based models, this is implemented via the ratio of negative to positive samples, while linear and ensemble baselines use built‑in balanced class weights. This approach preserves the original data distribution and produces probability estimates suitable for downstream calibration and cost‑based decision analysis.

In [5]:
X_train_, y_train = load_dataset("02", "train")
X_validation_, y_validation = load_dataset("02", "validation")


train
X shape: (28934, 60)
y shape: (28934,)

X head:


,cat__job_admin.,cat__job_blue-collar,cat__job_entrepreneur,cat__job_housemaid,cat__job_management,cat__job_retired,cat__job_self-employed,cat__job_services,cat__job_student,cat__job_technician,...,bin__housing,bin__loan,bin__poutcome_success,bin__contact_is_unknown,bin__poutcome_is_unknown,bin__multiple_contacts_current_campaign,bin__previous_and_many_current_contacts,bin__many_contacts_and_housing_loan,bin__both_housing_and_personal_loan,cluster
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,2
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0



validation
X shape: (7234, 60)
y shape: (7234,)

X head:


,cat__job_admin.,cat__job_blue-collar,cat__job_entrepreneur,cat__job_housemaid,cat__job_management,cat__job_retired,cat__job_self-employed,cat__job_services,cat__job_student,cat__job_technician,...,bin__housing,bin__loan,bin__poutcome_success,bin__contact_is_unknown,bin__poutcome_is_unknown,bin__multiple_contacts_current_campaign,bin__previous_and_many_current_contacts,bin__many_contacts_and_housing_loan,bin__both_housing_and_personal_loan,cluster
0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,3
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0
2,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0


In [6]:
cluster_feat = "cluster"

X_train = X_train_.copy()
X_train = X_train.drop(columns=cluster_feat)
print("\nTrain set: ", X_train.shape)
display(X_train.head(3))

X_validation = X_validation_.copy()
X_validation = X_validation.drop(columns=cluster_feat).copy()
print("\nValidation set: ", X_validation.shape)
display(X_validation.head(3))


Train set:  (28934, 59)


,cat__job_admin.,cat__job_blue-collar,cat__job_entrepreneur,cat__job_housemaid,cat__job_management,cat__job_retired,cat__job_self-employed,cat__job_services,cat__job_student,cat__job_technician,...,bin__default,bin__housing,bin__loan,bin__poutcome_success,bin__contact_is_unknown,bin__poutcome_is_unknown,bin__multiple_contacts_current_campaign,bin__previous_and_many_current_contacts,bin__many_contacts_and_housing_loan,bin__both_housing_and_personal_loan
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0



Validation set:  (7234, 59)


,cat__job_admin.,cat__job_blue-collar,cat__job_entrepreneur,cat__job_housemaid,cat__job_management,cat__job_retired,cat__job_self-employed,cat__job_services,cat__job_student,cat__job_technician,...,bin__default,bin__housing,bin__loan,bin__poutcome_success,bin__contact_is_unknown,bin__poutcome_is_unknown,bin__multiple_contacts_current_campaign,bin__previous_and_many_current_contacts,bin__many_contacts_and_housing_loan,bin__both_housing_and_personal_loan
0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0
2,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0


In [7]:
n_pos = y_train.sum()
n_neg = len(y_train) - n_pos
scale_pos_weight = n_neg / n_pos

### i. Logistic Regression

In [8]:
log_reg = LogisticRegression(
    penalty="l2",
    solver="liblinear",
    class_weight="balanced",
    max_iter=1000,
    random_state=RANDOM_STATE,
)

log_reg.fit(X_train, y_train)
log_reg_metrics = evaluate_model(log_reg, X_validation, y_validation)

### ii. Random Forest

In [9]:
rf = RandomForestClassifier(
    n_estimators=100, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
)

rf.fit(X_train, y_train)
rf_metrics = evaluate_model(rf, X_validation, y_validation)

### iii. XGBoost

In [10]:
xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
)

xgb.fit(X_train, y_train)
xgb_metrics = evaluate_model(xgb, X_validation, y_validation)

### iv. LightGBM

In [11]:
lgbm = lgb.LGBMClassifier(
    objective="binary", scale_pos_weight=scale_pos_weight, random_state=RANDOM_STATE
)

lgbm.fit(X_train, y_train)
lgbm_metrics = evaluate_model(lgbm, X_validation, y_validation)

[LightGBM] [Info] Number of positive: 3385, number of negative: 25549
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001607 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 176
[LightGBM] [Info] Number of data points in the train set: 28934, number of used features: 59
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.116990 -> initscore=-2.021244
[LightGBM] [Info] Start training from score -2.021244


### v. CatBoost

In [12]:
cat = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="PRAUC",
    class_weights=[1, scale_pos_weight],
    verbose=0,
    random_state=RANDOM_STATE,
)

cat.fit(X_train, y_train)
cat_metrics = evaluate_model(cat, X_validation, y_validation)

### Results comparison

In [13]:
results = pd.DataFrame.from_dict(
    {
        "Logistic Regression": log_reg_metrics,
        "Random Forest": rf_metrics,
        "XGBoost": xgb_metrics,
        "LightGBM": lgbm_metrics,
        "CatBoost": cat_metrics,
    },
    orient="index",
)

results.sort_values(by="AUC_PR", ascending=False)

,AUC_PR,F1,Precision,Recall
LightGBM,0.445832,0.459201,0.362826,0.625296
CatBoost,0.442484,0.459831,0.368159,0.612293
Logistic Regression,0.411748,0.395126,0.287325,0.632388
XGBoost,0.407729,0.434281,0.347795,0.578014
Random Forest,0.313136,0.275808,0.413712,0.206856


On the validation set, LightGBM and CatBoost exhibit very similar performance, with nearly identical F1 scores and only a marginal difference in AUC‑PR (0.446 for LightGBM versus 0.442 for CatBoost). Both models clearly outperform the other architectures and appear well suited to handling the strong class imbalance present in the data.

LightGBM was selected as the primary model architecture because it achieves the highest AUC‑PR overall, indicating slightly better ranking of likely subscribers, while maintaining strong recall and competitive precision. Given the minimal performance gap, this choice is pragmatic rather than definitive; CatBoost remains a strong alternative and could be expected to perform similarly in downstream analysis. **LightGBM** is therefore used for the remainder of the project to provide a consistent baseline for both global and segment‑specific modeling.

## 2. Global model

Following the architecture selection, we now develop a fully optimized Global Model. This model serves as the primary benchmark for the project, trained on the complete dataset to capture broad customer patterns. To manage the significant class imbalance (~11.7% success rate), we implement an analytical class-weighting strategy (scale_pos_weight) that penalizes errors on the minority class more heavily, ensuring the model prioritizes potential subscribers over the majority of non-subscribers.

Once we decided to work with LightGBM, we'll train the global model which will the observations from all clusters.

In [14]:
X_train_, y_train = load_dataset("02", "train")
X_validation_, y_validation = load_dataset("02", "validation")
# X_test_, y_test_ = load_dataset(nb="02", split="test")


train
X shape: (28934, 60)
y shape: (28934,)

X head:


,cat__job_admin.,cat__job_blue-collar,cat__job_entrepreneur,cat__job_housemaid,cat__job_management,cat__job_retired,cat__job_self-employed,cat__job_services,cat__job_student,cat__job_technician,...,bin__housing,bin__loan,bin__poutcome_success,bin__contact_is_unknown,bin__poutcome_is_unknown,bin__multiple_contacts_current_campaign,bin__previous_and_many_current_contacts,bin__many_contacts_and_housing_loan,bin__both_housing_and_personal_loan,cluster
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,2
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0



validation
X shape: (7234, 60)
y shape: (7234,)

X head:


,cat__job_admin.,cat__job_blue-collar,cat__job_entrepreneur,cat__job_housemaid,cat__job_management,cat__job_retired,cat__job_self-employed,cat__job_services,cat__job_student,cat__job_technician,...,bin__housing,bin__loan,bin__poutcome_success,bin__contact_is_unknown,bin__poutcome_is_unknown,bin__multiple_contacts_current_campaign,bin__previous_and_many_current_contacts,bin__many_contacts_and_housing_loan,bin__both_housing_and_personal_loan,cluster
0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,3
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0
2,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0


In [15]:
cluster_feat = "cluster"

X_train = X_train_.copy()
X_train = X_train.drop(columns=cluster_feat)
print("\nTrain set: ", X_train.shape)
display(X_train.head(3))

X_validation = X_validation_.copy()
X_validation = X_validation.drop(columns=cluster_feat).copy()
print("\nValidation set: ", X_validation.shape)
display(X_validation.head(3))

# X_test = X_test_.copy()
# X_test = X_test.drop(columns=cluster_feat).copy()
# print("\nTest set: ", X_test.shape)
# display(X_test.head(3))


Train set:  (28934, 59)


,cat__job_admin.,cat__job_blue-collar,cat__job_entrepreneur,cat__job_housemaid,cat__job_management,cat__job_retired,cat__job_self-employed,cat__job_services,cat__job_student,cat__job_technician,...,bin__default,bin__housing,bin__loan,bin__poutcome_success,bin__contact_is_unknown,bin__poutcome_is_unknown,bin__multiple_contacts_current_campaign,bin__previous_and_many_current_contacts,bin__many_contacts_and_housing_loan,bin__both_housing_and_personal_loan
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0



Validation set:  (7234, 59)


,cat__job_admin.,cat__job_blue-collar,cat__job_entrepreneur,cat__job_housemaid,cat__job_management,cat__job_retired,cat__job_self-employed,cat__job_services,cat__job_student,cat__job_technician,...,bin__default,bin__housing,bin__loan,bin__poutcome_success,bin__contact_is_unknown,bin__poutcome_is_unknown,bin__multiple_contacts_current_campaign,bin__previous_and_many_current_contacts,bin__many_contacts_and_housing_loan,bin__both_housing_and_personal_loan
0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0
2,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0


In [16]:
n_pos = y_train.sum()
n_neg = len(y_train) - n_pos
scale_pos_weight = n_neg / n_pos

print(f"Scale pos weight: {scale_pos_weight:.2f}")

Scale pos weight: 7.55


### 1. Hyperparameter Optimization via Optuna

We employ Bayesian Optimization using the Optuna framework to fine-tune the LightGBM hyperparameters. Unlike a standard grid search, this approach efficiently explores the parameter space to maximize the Area Under the Precision-Recall Curve (AUC-PR). This specific metric is chosen to ensure high model reliability and ranking performance within the minority positive class, which is essential for effective telemarketing targeting.

In [17]:
def objective(trial):
    params = {
        "objective": "binary",
        "metric": "aucpr",
        "random_state": RANDOM_STATE,
        "scale_pos_weight": scale_pos_weight,
        "verbosity": -1,
        # hyperparameters to tune
        "num_leaves": trial.suggest_int("num_leaves", 16, 128),
        "max_depth": trial.suggest_int("max_depth", -1, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 200, 600),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 200),
    }

    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train)

    y_val_prob = model.predict_proba(X_validation)[:, 1]
    aucpr = average_precision_score(y_validation, y_val_prob)

    return aucpr

In [18]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30, show_progress_bar=True)

print("Best AUC-PR:", study.best_value)
print("Best parameters:", study.best_params)

best_params = study.best_params

[I 2026-04-11 16:13:08,449] A new study created in memory with name: no-name-eeb3761b-2491-4c76-b9e3-f8c79b012502


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-04-11 16:13:09,756] Trial 0 finished with value: 0.4470332058032621 and parameters: {'num_leaves': 27, 'max_depth': 0, 'learning_rate': 0.06622271315830758, 'n_estimators': 401, 'min_child_samples': 183}. Best is trial 0 with value: 0.4470332058032621.
[I 2026-04-11 16:13:10,452] Trial 1 finished with value: 0.4744939271188465 and parameters: {'num_leaves': 67, 'max_depth': 10, 'learning_rate': 0.029138622860095602, 'n_estimators': 223, 'min_child_samples': 188}. Best is trial 1 with value: 0.4744939271188465.
[I 2026-04-11 16:13:11,114] Trial 2 finished with value: 0.48427836775721544 and parameters: {'num_leaves': 80, 'max_depth': 5, 'learning_rate': 0.03097018420223754, 'n_estimators': 368, 'min_child_samples': 39}. Best is trial 2 with value: 0.48427836775721544.
[I 2026-04-11 16:13:11,246] Trial 3 finished with value: 0.4489115886708086 and parameters: {'num_leaves': 96, 'max_depth': 1, 'learning_rate': 0.07268337434343158, 'n_estimators': 404, 'min_child_samples': 34}. Be

We train the final global base model with the best parameters

In [ ]:
global_model = lgb.LGBMClassifier(
    objective="binary",
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    **best_params
)

global_model.fit(X_train, y_train)

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,4
,learning_rate,0.09908788645704905
,n_estimators,348
,subsample_for_bin,200000
,objective,'binary'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,55


### 2. Probability Calibration

Machine learning models, particularly those using class-weighting, often produce "scores" rather than true probabilities. We apply Isotonic Regression to calibrate these outputs, aligning the model’s predicted likelihoods with the actual observed subscription rates. Well-calibrated probabilities are a prerequisite for our upcoming Business Cost Framework, where we will calculate expected ROI based on specific call costs and subscription values.

In [ ]:
# raw probabilities on validation
val_probs_raw = global_model.predict_proba(X_validation)[:, 1]

# isotonic regression calibration
calibrator = IsotonicRegression(out_of_bounds="clip")
calibrator.fit(val_probs_raw, y_validation)

# calibrated probabilities
val_probs_calibrated = calibrator.transform(val_probs_raw)

In [ ]:
global_validation_metrics = evaluate_from_probs(y_validation, val_probs_calibrated)

print("Global model (validation, calibrated):")
print(global_validation_metrics)

Global model (validation, calibrated):
{'AUC_PR': 0.4457539936060022, 'F1': 0.2846441947565543, 'Precision': 0.6846846846846847, 'Recall': 0.17966903073286053}


### 3. Baseline Performance across Customer Segments

To evaluate the "one-size-fits-all" approach, we project the Global Model’s performance onto our four distinct customer clusters. This diagnostic step identifies where the global patterns are robust and where they fail to capture localized nuances.

In [ ]:
cluster_results = {}

for c in sorted(X_validation_[cluster_feat].unique()):
    idx = X_validation_[cluster_feat] == c

    cluster_results[f"cluster_{c}"] = evaluate_from_probs(
        y_validation.loc[idx], val_probs_calibrated[idx]
    )

cluster_results_df = pd.DataFrame(cluster_results).T
cluster_results_df

,AUC_PR,F1,Precision,Recall
cluster_0,0.563676,0.387597,0.757576,0.260417
cluster_1,0.295621,0.171717,0.629630,0.099415
cluster_2,0.380164,0.243478,0.608696,0.152174
cluster_3,0.495403,0.298429,0.686747,0.190635


In [ ]:
model_dir = Path("models")

model_name = "global"

# save base model
joblib.dump(global_model, model_dir / f"{model_name}_model.joblib")

# save calibrator
joblib.dump(calibrator, model_dir / f"{model_name}_calibrator.joblib")

# save metadata
metadata = {
    "best_params": best_params,
    "scale_pos_weight": scale_pos_weight,
    "features": list(X_train.columns),
    "random_state": RANDOM_STATE,
}

joblib.dump(metadata, model_dir / f"{model_name}_metadata.joblib")

['models/global_metadata.joblib']

## 3. Cluster 0 model

In [ ]:
cluster_id = 0
X_train, y_train = load_dataset(nb="02", split="train", cluster_id=cluster_id)
X_validation, y_validation = load_dataset(
    nb="02", split="validation", cluster_id=cluster_id
)
# X_test, y_test = load_dataset(nb="02", split="test", cluster_id=cluster_id)


train | cluster 0
X shape: (5043, 59)
y shape: (5043,)

X head:


,cat__job_admin.,cat__job_blue-collar,cat__job_entrepreneur,cat__job_housemaid,cat__job_management,cat__job_retired,cat__job_self-employed,cat__job_services,cat__job_student,cat__job_technician,...,bin__default,bin__housing,bin__loan,bin__poutcome_success,bin__contact_is_unknown,bin__poutcome_is_unknown,bin__multiple_contacts_current_campaign,bin__previous_and_many_current_contacts,bin__many_contacts_and_housing_loan,bin__both_housing_and_personal_loan
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0
6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0
10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0



validation | cluster 0
X shape: (1271, 59)
y shape: (1271,)

X head:


,cat__job_admin.,cat__job_blue-collar,cat__job_entrepreneur,cat__job_housemaid,cat__job_management,cat__job_retired,cat__job_self-employed,cat__job_services,cat__job_student,cat__job_technician,...,bin__default,bin__housing,bin__loan,bin__poutcome_success,bin__contact_is_unknown,bin__poutcome_is_unknown,bin__multiple_contacts_current_campaign,bin__previous_and_many_current_contacts,bin__many_contacts_and_housing_loan,bin__both_housing_and_personal_loan
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0
2,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0


In [ ]:
# 2. Calculate the specific scale_pos_weight for Cluster 0
# Segment-specific weighting is essential as subscription rates vary by cluster
n_pos = y_train.sum()
n_neg = len(y_train) - n_pos
scale_pos_weight = n_neg / n_pos

print(f"Cluster {cluster_id} - Training Shape: {X_train.shape}")
print(f"Cluster {cluster_id} - Scale Pos Weight: {scale_pos_weight:.2f}")

Cluster 0 - Training Shape: (5043, 59)
Cluster 0 - Scale Pos Weight: 6.14


### 3.1. Hyperparameter Optimization via Optuna

In [ ]:
def objective(trial):
    params = {
        "objective": "binary",
        "metric": "aucpr",
        "random_state": RANDOM_STATE,
        "scale_pos_weight": scale_pos_weight,
        "verbosity": -1,
        # hyperparameters to tune
        "num_leaves": trial.suggest_int("num_leaves", 16, 128),
        "max_depth": trial.suggest_int("max_depth", -1, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 200, 600),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 200),
    }

    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train)

    y_val_prob = model.predict_proba(X_validation)[:, 1]
    aucpr = average_precision_score(y_validation, y_val_prob)

    return aucpr

In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30, show_progress_bar=True)

print("Best AUC-PR:", study.best_value)
print("Best parameters:", study.best_params)

best_params = study.best_params

[I 2026-04-11 07:28:57,585] A new study created in memory with name: no-name-8e38ff80-4890-4747-84d9-82b21c84db51
Best trial: 0. Best value: 0.535751:   3%|▎         | 1/30 [00:00<00:12,  2.32it/s]

[I 2026-04-11 07:28:58,026] Trial 0 finished with value: 0.5357513535198656 and parameters: {'num_leaves': 105, 'max_depth': 3, 'learning_rate': 0.020210533861844435, 'n_estimators': 286, 'min_child_samples': 91}. Best is trial 0 with value: 0.5357513535198656.


Best trial: 0. Best value: 0.535751:   7%|▋         | 2/30 [00:02<00:41,  1.50s/it]

[I 2026-04-11 07:29:00,268] Trial 1 finished with value: 0.5240057342779578 and parameters: {'num_leaves': 53, 'max_depth': 9, 'learning_rate': 0.048503408146279015, 'n_estimators': 397, 'min_child_samples': 96}. Best is trial 0 with value: 0.5357513535198656.


Best trial: 2. Best value: 0.550214:  10%|█         | 3/30 [00:04<00:47,  1.77s/it]

...

[I 2026-04-11 07:29:52,732] Trial 27 finished with value: 0.5298370633139018 and parameters: {'num_leaves': 24, 'max_depth': 11, 'learning_rate': 0.05891838528043975, 'n_estimators': 572, 'min_child_samples': 165}. Best is trial 12 with value: 0.5532474225944112.


Best trial: 12. Best value: 0.553247:  97%|█████████▋| 29/30 [00:57<00:02,  2.22s/it]

[I 2026-04-11 07:29:54,720] Trial 28 finished with value: 0.5453560467966511 and parameters: {'num_leaves': 43, 'max_depth': 8, 'learning_rate': 0.02493919954169482, 'n_estimators': 530, 'min_child_samples': 184}. Best is trial 12 with value: 0.5532474225944112.


Best trial: 12. Best value: 0.553247: 100%|██████████| 30/30 [00:57<00:00,  1.92s/it]

[I 2026-04-11 07:29:55,191] Trial 29 finished with value: 0.5429181306706922 and parameters: {'num_leaves': 78, 'max_depth': 3, 'learning_rate': 0.04102745775483835, 'n_estimators': 313, 'min_child_samples': 75}. Best is trial 12 with value: 0.5532474225944112.
Best AUC-PR: 0.5532474225944112
Best parameters: {'num_leaves': 16, 'max_depth': -1, 'learning_rate': 0.030612566719765602, 'n_estimators': 595, 'min_child_samples': 171}


In [ ]:
model = lgb.LGBMClassifier(
    objective="binary",
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    **best_params
)

model.fit(X_train, y_train)

,boosting_type,'gbdt'
,num_leaves,16
,max_depth,-1
,learning_rate,0.030612566719765602
,n_estimators,595
,subsample_for_bin,200000
,objective,'binary'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,171


### 3.2. Probability Calibration

In [ ]:
# raw probabilities on validation
val_probs_raw = model.predict_proba(X_validation)[:, 1]

# isotonic regression calibration
calibrator = IsotonicRegression(out_of_bounds="clip")
calibrator.fit(val_probs_raw, y_validation)

# calibrated probabilities
val_probs_calibrated = calibrator.transform(val_probs_raw)

In [ ]:
c0_validation_metrics = evaluate_from_probs(y_validation, val_probs_calibrated)

print(f"Cluster {cluster_id} model (validation, calibrated):")
print(c0_validation_metrics)

Cluster 0 model (validation, calibrated):
{'AUC_PR': 0.553079139992169, 'F1': 0.4198473282442748, 'Precision': 0.7857142857142857, 'Recall': 0.2864583333333333}


We save the model

In [ ]:
model_dir = Path("models")

model_name = f"c{cluster_id}"

# save base model
joblib.dump(model, model_dir / f"{model_name}_model.joblib")

# save calibrator
joblib.dump(calibrator, model_dir / f"{model_name}_calibrator.joblib")

# save metadata
metadata = {
    "best_params": best_params,
    "scale_pos_weight": scale_pos_weight,
    "features": list(X_train.columns),
    "random_state": RANDOM_STATE,
}

joblib.dump(metadata, model_dir / f"{model_name}_metadata.joblib")

['models/c0_metadata.joblib']

## 4. Cluster 1 model

In [ ]:
cluster_id = 1
X_train, y_train = load_dataset(nb="02", split="train", cluster_id=cluster_id)
X_validation, y_validation = load_dataset(
    nb="02", split="validation", cluster_id=cluster_id
)
X_test, y_test = load_dataset(nb="02", split="test", cluster_id=cluster_id)

In [ ]:
# 2. Calculate the specific scale_pos_weight for Cluster 1
n_pos = y_train.sum()
n_neg = len(y_train) - n_pos
scale_pos_weight = n_neg / n_pos

print(f"Cluster {cluster_id} - Training Shape: {X_train.shape}")
print(f"Cluster {cluster_id} - Scale Pos Weight: {scale_pos_weight:.2f}")


### 4.1. Hyperparameter Optimization via Optuna


In [ ]:
def objective(trial):
    params = {
        "objective": "binary",
        "metric": "aucpr",
        "random_state": RANDOM_STATE,
        "scale_pos_weight": scale_pos_weight,
        "verbosity": -1,
        "num_leaves": trial.suggest_int("num_leaves", 16, 128),
        "max_depth": trial.suggest_int("max_depth", -1, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 200, 600),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 200),
    }

    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train)

    y_val_prob = model.predict_proba(X_validation)[:, 1]
    aucpr = average_precision_score(y_validation, y_val_prob)
    return aucpr


In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30, show_progress_bar=True)

print("Best AUC-PR:", study.best_value)
print("Best parameters:", study.best_params)

best_params = study.best_params


In [ ]:
model = lgb.LGBMClassifier(
    objective="binary",
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    **best_params
)

model.fit(X_train, y_train)


### 4.2. Probability Calibration


In [ ]:
# raw probabilities on validation
val_probs_raw = model.predict_proba(X_validation)[:, 1]

# isotonic regression calibration
calibrator = IsotonicRegression(out_of_bounds="clip")
calibrator.fit(val_probs_raw, y_validation)

# calibrated probabilities
val_probs_calibrated = calibrator.transform(val_probs_raw)


In [ ]:
c1_validation_metrics = evaluate_from_probs(y_validation, val_probs_calibrated)

print(f"Cluster {cluster_id} model (validation, calibrated):")
print(c1_validation_metrics)


In [ ]:
# save the model and calibrator for cluster 1
model_dir = Path("models")
model_name = f"c{cluster_id}"

# save model
joblib.dump(model, model_dir / f"{model_name}_model.joblib")

# save calibrator
joblib.dump(calibrator, model_dir / f"{model_name}_calibrator.joblib")

# save metadata
metadata = {
    "best_params": best_params,
    "scale_pos_weight": scale_pos_weight,
    "features": list(X_train.columns),
    "random_state": RANDOM_STATE,
}

joblib.dump(metadata, model_dir / f"{model_name}_metadata.joblib")


## 5. Cluster 2 model

In [19]:
cluster_id = 2
X_train, y_train = load_dataset(nb="02", split="train", cluster_id=cluster_id)
X_validation, y_validation = load_dataset(
    nb="02", split="validation", cluster_id=cluster_id
)
X_test, y_test = load_dataset(nb="02", split="test", cluster_id=cluster_id)


train | cluster 2
X shape: (7058, 59)
y shape: (7058,)

X head:


,cat__job_admin.,cat__job_blue-collar,cat__job_entrepreneur,cat__job_housemaid,cat__job_management,cat__job_retired,cat__job_self-employed,cat__job_services,cat__job_student,cat__job_technician,...,bin__default,bin__housing,bin__loan,bin__poutcome_success,bin__contact_is_unknown,bin__poutcome_is_unknown,bin__multiple_contacts_current_campaign,bin__previous_and_many_current_contacts,bin__many_contacts_and_housing_loan,bin__both_housing_and_personal_loan
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0
11,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0
12,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0



validation | cluster 2
X shape: (1720, 59)
y shape: (1720,)

X head:


,cat__job_admin.,cat__job_blue-collar,cat__job_entrepreneur,cat__job_housemaid,cat__job_management,cat__job_retired,cat__job_self-employed,cat__job_services,cat__job_student,cat__job_technician,...,bin__default,bin__housing,bin__loan,bin__poutcome_success,bin__contact_is_unknown,bin__poutcome_is_unknown,bin__multiple_contacts_current_campaign,bin__previous_and_many_current_contacts,bin__many_contacts_and_housing_loan,bin__both_housing_and_personal_loan
8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0
11,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0



test | cluster 2
X shape: (2201, 59)
y shape: (2201,)

X head:


,cat__job_admin.,cat__job_blue-collar,cat__job_entrepreneur,cat__job_housemaid,cat__job_management,cat__job_retired,cat__job_self-employed,cat__job_services,cat__job_student,cat__job_technician,...,bin__default,bin__housing,bin__loan,bin__poutcome_success,bin__contact_is_unknown,bin__poutcome_is_unknown,bin__multiple_contacts_current_campaign,bin__previous_and_many_current_contacts,bin__many_contacts_and_housing_loan,bin__both_housing_and_personal_loan
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0
13,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0


In [20]:
# 2. Calculate the specific scale_pos_weight for Cluster 2
n_pos = y_train.sum()
n_neg = len(y_train) - n_pos
scale_pos_weight = n_neg / n_pos

print(f"Cluster {cluster_id} - Training Shape: {X_train.shape}")
print(f"Cluster {cluster_id} - Scale Pos Weight: {scale_pos_weight:.2f}")

Cluster 2 - Training Shape: (7058, 59)
Cluster 2 - Scale Pos Weight: 7.96


### 5.1. Hyperparameter Optimization via Optuna

In [21]:
def objective(trial):
    params = {
        "objective": "binary",
        "metric": "aucpr",
        "random_state": RANDOM_STATE,
        "scale_pos_weight": scale_pos_weight,
        "verbosity": -1,
        "num_leaves": trial.suggest_int("num_leaves", 16, 128),
        "max_depth": trial.suggest_int("max_depth", -1, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 200, 600),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 200),
    }

    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train)

    y_val_prob = model.predict_proba(X_validation)[:, 1]
    aucpr = average_precision_score(y_validation, y_val_prob)
    return aucpr

In [22]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30, show_progress_bar=True)

print("Best AUC-PR:", study.best_value)
print("Best parameters:", study.best_params)

best_params = study.best_params

[I 2026-04-11 15:46:05,317] A new study created in memory with name: no-name-76d0a877-77e7-42a0-89b4-6d4408db74c1


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-04-11 15:46:06,621] Trial 0 finished with value: 0.33045298241501103 and parameters: {'num_leaves': 65, 'max_depth': 7, 'learning_rate': 0.03230630888859284, 'n_estimators': 591, 'min_child_samples': 149}. Best is trial 0 with value: 0.33045298241501103.
[I 2026-04-11 15:46:06,874] Trial 1 finished with value: 0.30311354360654935 and parameters: {'num_leaves': 35, 'max_depth': 4, 'learning_rate': 0.1046326652386573, 'n_estimators': 230, 'min_child_samples': 187}. Best is trial 0 with value: 0.33045298241501103.
[I 2026-04-11 15:46:07,739] Trial 2 finished with value: 0.3571394055854472 and parameters: {'num_leaves': 17, 'max_depth': 10, 'learning_rate': 0.01971329821735484, 'n_estimators': 419, 'min_child_samples': 150}. Best is trial 2 with value: 0.3571394055854472.
[I 2026-04-11 15:46:08,599] Trial 3 finished with value: 0.33151207121078746 and parameters: {'num_leaves': 78, 'max_depth': 0, 'learning_rate': 0.053162637445759185, 'n_estimators': 213, 'min_child_samples': 150}

In [23]:
model = lgb.LGBMClassifier(
    objective="binary",
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    **best_params
)

model.fit(X_train, y_train)

,boosting_type,'gbdt'
,num_leaves,116
,max_depth,4
,learning_rate,0.01796046556719597
,n_estimators,426
,subsample_for_bin,200000
,objective,'binary'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,32


### 5.2. Probability Calibration

In [24]:
# raw probabilities on validation
val_probs_raw = model.predict_proba(X_validation)[:, 1]

# isotonic regression calibration
calibrator = IsotonicRegression(out_of_bounds="clip")
calibrator.fit(val_probs_raw, y_validation)

# calibrated probabilities
val_probs_calibrated = calibrator.transform(val_probs_raw)

In [25]:
c2_validation_metrics = evaluate_from_probs(y_validation, val_probs_calibrated)

print(f"Cluster {cluster_id} model (validation, calibrated):")
print(c2_validation_metrics)

Cluster 2 model (validation, calibrated):
{'AUC_PR': 0.35514012731149774, 'F1': 0.27074235807860264, 'Precision': 0.6888888888888889, 'Recall': 0.16847826086956522}


In [ ]:
# save the model and calibrator for cluster 2
model_dir = Path("models")
model_name = f"c{cluster_id}"

# save model
joblib.dump(model, model_dir / f"{model_name}_model.joblib")

# save calibrator
joblib.dump(calibrator, model_dir / f"{model_name}_calibrator.joblib")

# save metadata
metadata = {
    "best_params": best_params,
    "scale_pos_weight": scale_pos_weight,
    "features": list(X_train.columns),
    "random_state": RANDOM_STATE,
}

joblib.dump(metadata, model_dir / f"{model_name}_metadata.joblib")

['models/c2_metadata.joblib']

## 6. Cluster 3 model

In [27]:
cluster_id = 3
X_train, y_train = load_dataset(nb="02", split="train", cluster_id=cluster_id)
X_validation, y_validation = load_dataset(
    nb="02", split="validation", cluster_id=cluster_id
)
X_test, y_test = load_dataset(nb="02", split="test", cluster_id=cluster_id)


train | cluster 3
X shape: (7986, 59)
y shape: (7986,)

X head:


,cat__job_admin.,cat__job_blue-collar,cat__job_entrepreneur,cat__job_housemaid,cat__job_management,cat__job_retired,cat__job_self-employed,cat__job_services,cat__job_student,cat__job_technician,...,bin__default,bin__housing,bin__loan,bin__poutcome_success,bin__contact_is_unknown,bin__poutcome_is_unknown,bin__multiple_contacts_current_campaign,bin__previous_and_many_current_contacts,bin__many_contacts_and_housing_loan,bin__both_housing_and_personal_loan
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0
8,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0
9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



validation | cluster 3
X shape: (1977, 59)
y shape: (1977,)

X head:


,cat__job_admin.,cat__job_blue-collar,cat__job_entrepreneur,cat__job_housemaid,cat__job_management,cat__job_retired,cat__job_self-employed,cat__job_services,cat__job_student,cat__job_technician,...,bin__default,bin__housing,bin__loan,bin__poutcome_success,bin__contact_is_unknown,bin__poutcome_is_unknown,bin__multiple_contacts_current_campaign,bin__previous_and_many_current_contacts,bin__many_contacts_and_housing_loan,bin__both_housing_and_personal_loan
0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
6,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
12,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0



test | cluster 3
X shape: (2461, 59)
y shape: (2461,)

X head:


,cat__job_admin.,cat__job_blue-collar,cat__job_entrepreneur,cat__job_housemaid,cat__job_management,cat__job_retired,cat__job_self-employed,cat__job_services,cat__job_student,cat__job_technician,...,bin__default,bin__housing,bin__loan,bin__poutcome_success,bin__contact_is_unknown,bin__poutcome_is_unknown,bin__multiple_contacts_current_campaign,bin__previous_and_many_current_contacts,bin__many_contacts_and_housing_loan,bin__both_housing_and_personal_loan
0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
9,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0


In [28]:
# Calculate cluster-specific scale_pos_weight
n_pos = y_train.sum()
n_neg = len(y_train) - n_pos
scale_pos_weight = n_neg / n_pos

print(f"Cluster {cluster_id} - Training Shape: {X_train.shape}")
print(f"Cluster {cluster_id} - Scale Pos Weight: {scale_pos_weight:.2f}")

Cluster 3 - Training Shape: (7986, 59)
Cluster 3 - Scale Pos Weight: 5.80


### 6.1. Hyperparameter Optimization via Optuna


In [29]:
def objective(trial):
    params = {
        "objective": "binary",
        "metric": "aucpr",
        "random_state": RANDOM_STATE,
        "scale_pos_weight": scale_pos_weight,
        "verbosity": -1,
        "num_leaves": trial.suggest_int("num_leaves", 16, 128),
        "max_depth": trial.suggest_int("max_depth", -1, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 200, 600),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 200),
    }

    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train)

    y_val_prob = model.predict_proba(X_validation)[:, 1]
    aucpr = average_precision_score(y_validation, y_val_prob)
    return aucpr

In [30]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30, show_progress_bar=True)

print("Best AUC-PR:", study.best_value)
print("Best parameters:", study.best_params)

best_params = study.best_params

[I 2026-04-11 15:57:09,132] A new study created in memory with name: no-name-da2a2176-13c4-457e-a81e-d633a5b80b03


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-04-11 15:57:10,431] Trial 0 finished with value: 0.42263663267642987 and parameters: {'num_leaves': 75, 'max_depth': 7, 'learning_rate': 0.13901086465606127, 'n_estimators': 438, 'min_child_samples': 45}. Best is trial 0 with value: 0.42263663267642987.
[I 2026-04-11 15:57:10,618] Trial 1 finished with value: 0.4590866716212569 and parameters: {'num_leaves': 75, 'max_depth': 2, 'learning_rate': 0.01361619265723939, 'n_estimators': 392, 'min_child_samples': 96}. Best is trial 1 with value: 0.4590866716212569.
[I 2026-04-11 15:57:13,428] Trial 2 finished with value: 0.40441157595385746 and parameters: {'num_leaves': 63, 'max_depth': 11, 'learning_rate': 0.1372648384420804, 'n_estimators': 560, 'min_child_samples': 64}. Best is trial 1 with value: 0.4590866716212569.
[I 2026-04-11 15:57:14,412] Trial 3 finished with value: 0.4777140837365468 and parameters: {'num_leaves': 41, 'max_depth': 0, 'learning_rate': 0.03785140719922984, 'n_estimators': 207, 'min_child_samples': 80}. Best 

In [31]:
model = lgb.LGBMClassifier(
    objective="binary",
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    **best_params
)

model.fit(X_train, y_train)

,boosting_type,'gbdt'
,num_leaves,57
,max_depth,4
,learning_rate,0.01597898484897106
,n_estimators,426
,subsample_for_bin,200000
,objective,'binary'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,102


### 6.2. Probability Calibration


In [32]:
# raw probabilities on validation
val_probs_raw = model.predict_proba(X_validation)[:, 1]

# isotonic regression calibration
calibrator = IsotonicRegression(out_of_bounds="clip")
calibrator.fit(val_probs_raw, y_validation)

# calibrated probabilities
val_probs_calibrated = calibrator.transform(val_probs_raw)

In [33]:
c3_validation_metrics = evaluate_from_probs(y_validation, val_probs_calibrated)

print(f"Cluster {cluster_id} model (validation, calibrated):")
print(c3_validation_metrics)

Cluster 3 model (validation, calibrated):
{'AUC_PR': 0.4800591719450929, 'F1': 0.42462845010615713, 'Precision': 0.5813953488372093, 'Recall': 0.33444816053511706}


In [ ]:
# save the model and calibrator for cluster 3
model_dir = Path("models")
model_name = f"c{cluster_id}"

# save model
joblib.dump(model, model_dir / f"{model_name}_model.joblib")

# save calibrator
joblib.dump(calibrator, model_dir / f"{model_name}_calibrator.joblib")

# save metadata
metadata = {
    "best_params": best_params,
    "scale_pos_weight": scale_pos_weight,
    "features": list(X_train.columns),
    "random_state": RANDOM_STATE,
}

joblib.dump(metadata, model_dir / f"{model_name}_metadata.joblib")

['models/c3_metadata.joblib']